# bob, explained - Episode 10: Bitstream generation

Bitstream generation

Run `!pip install manim` and `from manim import *` once first, then this cell. Start at `-ql`; the class names listed at the top of the cell render a single section.


In [ ]:
%%manim -qm Ep10Bitgen
# =============================================================================
#  bob, explained - EPISODE 10: Bitstream generation
#  Bitstream generation
#
#  GENERATED by docs/manim/build.py from docs/manim/parts/. Do not edit here.
#
#  Prerequisite (once per notebook, in a cell of its own):
#      !pip install manim
#      from manim import *
#
#  Quality on the magic line above:  -ql draft   -qm medium   -qh 1080p60
#
#  Render one section instead of the whole episode by putting any of these
#  class names on the magic line:
#      E10S1Fasm
#      E10S2Bitgen
#      E10S3Bitfile
#      E10S4Load
#      E10S5Files
# =============================================================================

# =============================================================================
#  shared prelude - palette, helpers and the BobScene base class.
#  docs/manim/build.py pastes this into the top of every episode cell.
# =============================================================================

from manim import *
import numpy as np

# ---------------------------------------------------------------- palette ----
BG    = "#11121a"
INK   = "#e8e8ea"
DIM   = "#8b93a7"
C_PY  = "#7aa2f7"   # blue    - Python / tools / the device description
C_VPR = "#f7768e"   # red     - VPR / external tools
C_RTL = "#9ece6a"   # green   - hardware, Verilog, things on the die
C_BIT = "#e0af68"   # amber   - configuration bits, FASM, the bitstream
C_GRF = "#bb9af7"   # purple  - graphs, JTAG, protocol
C_ERR = "#ff7a93"   # pink    - bugs, refusals, errors
MONO  = "monospace"


# ---------------------------------------------------------------- helpers ----
def mono(s, size=22, color=INK):
    """One line of monospace text (Pango crashes on '', so blanks become ' ')."""
    return Text(s if s else " ", font=MONO, font_size=size, color=color)


def code_block(lines, size=20, color=INK):
    g = VGroup(*[mono(l, size, color) for l in lines])
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.14)
    return g


def panel(mob, color=DIM, pad=0.32, fill=0.06):
    r = SurroundingRectangle(mob, color=color, buff=pad)
    r.set_fill(color, opacity=fill)
    return VGroup(r, mob)


def chip(label, color, w=2.6, h=0.95, size=22, weight="NORMAL"):
    box = RoundedRectangle(width=w, height=h, corner_radius=0.14,
                           color=color, stroke_width=3)
    box.set_fill(color, opacity=0.12)
    txt = Text(label, font_size=size, color=INK, weight=weight, line_spacing=0.75)
    if txt.width > w - 0.3:
        txt.scale_to_fit_width(w - 0.3)
    if txt.height > h - 0.2:
        txt.scale_to_fit_height(h - 0.2)
    return VGroup(box, txt.move_to(box.get_center()))


def arrow(a, b, color=DIM, buff=0.15, sw=3):
    return Arrow(a, b, buff=buff, color=color, stroke_width=sw,
                 max_tip_length_to_length_ratio=0.18)


def mux_symbol(color=C_RTL, h=1.9, w=0.8):
    """Classic trapezoid multiplexer symbol."""
    p = Polygon([-w / 2,  h / 2, 0], [w / 2,  h / 2 - 0.3, 0],
                [ w / 2, -h / 2 + 0.3, 0], [-w / 2, -h / 2, 0],
                color=color, stroke_width=3)
    p.set_fill(color, opacity=0.14)
    return p


def bitcells(n, size=0.3, on=(), color=C_BIT, off_color=DIM):
    """A strip of n little squares; indices in `on` are filled."""
    g = VGroup()
    for i in range(n):
        s = Square(size, color=off_color, stroke_width=1.6)
        if i in on:
            s.set_stroke(color).set_fill(color, opacity=0.85)
        g.add(s)
    g.arrange(RIGHT, buff=0.035)
    return g


def fieldbar(fields, total_w=11.0, h=0.62, size=15):
    """
    fields: [(label, nbits, color), ...] -> one horizontal bar split to scale,
    each slice labelled above and its bit range below. Returns VGroup(bar, labels, ranges).
    """
    nbits = sum(f[1] for f in fields)
    bar, labs, rngs = VGroup(), VGroup(), VGroup()
    x, lo = -total_w / 2, 0
    for label, n, col in fields:
        w = max(total_w * n / nbits, 0.34)
        r = Rectangle(width=w, height=h, color=col, stroke_width=2)
        r.set_fill(col, opacity=0.28).move_to(np.array([x + w / 2, 0, 0]))
        bar.add(r)
        t = Text(label, font_size=size, color=col)
        if t.width > w * 1.9:
            t.scale_to_fit_width(max(w * 1.9, 0.5))
        t.next_to(r, UP, buff=0.14)
        labs.add(t)
        rt = mono(f"{lo}" if n == 1 else f"{lo}..{lo + n - 1}", size - 2, DIM)
        rt.next_to(r, DOWN, buff=0.12)
        if rt.width > w * 1.9:
            rt.scale_to_fit_width(max(w * 1.9, 0.5))
        rngs.add(rt)
        x += w
        lo += n
    return VGroup(bar, labs, rngs)


def filecard(path, role, color):
    """A small card naming a repo file and what it is."""
    t = mono(path, 17, color)
    r = Text(role, font_size=14, color=DIM)
    g = VGroup(t, r).arrange(DOWN, aligned_edge=LEFT, buff=0.08)
    box = SurroundingRectangle(g, color=color, buff=0.16)
    box.set_fill(color, opacity=0.07)
    return VGroup(box, g)


def mid(a, b):
    """midpoint, defined here so nothing depends on manim exporting space_ops."""
    return (a + b) / 2


def clear_all(sc, run_time=0.6):
    if sc.mobjects:
        sc.play(*[FadeOut(m) for m in sc.mobjects], run_time=run_time)


class BobScene(Scene):
    def setup(self):
        self.camera.background_color = BG

    def heading(self, text, kicker=None):
        t = Text(text, font_size=32, color=INK, weight="BOLD")
        t.to_corner(UL).shift(DOWN * 0.1)
        rule = Line(LEFT * 6.6, RIGHT * 6.6, color=DIM, stroke_width=1.5)
        rule.next_to(t, DOWN, buff=0.2).align_to(t, LEFT)
        g = VGroup(t, rule)
        self.play(FadeIn(t, shift=RIGHT * 0.3), Create(rule), run_time=0.7)
        if kicker:
            k = Text(kicker, font_size=19, color=DIM)
            if k.width > 13.0:
                k.scale_to_fit_width(13.0)
            k.next_to(rule, DOWN, buff=0.16).align_to(t, LEFT)
            g.add(k)
            self.play(FadeIn(k), run_time=0.4)
        return g

    def titlecard(self, number, title, subtitle):
        n = Text(number, font_size=26, color=C_BIT, weight="BOLD")
        t = Text(title, font_size=60, color=INK, weight="BOLD")
        s = Text(subtitle, font_size=26, color=DIM)
        if t.width > 12.5:
            t.scale_to_fit_width(12.5)
        if s.width > 12.5:
            s.scale_to_fit_width(12.5)
        g = VGroup(n, t, s).arrange(DOWN, buff=0.4)
        self.play(FadeIn(n), run_time=0.4)
        self.play(Write(t), run_time=1.1)
        self.play(FadeIn(s, shift=UP * 0.2), run_time=0.7)
        self.wait(1.6)
        self.play(FadeOut(g), run_time=0.6)

    def files_used(self, inputs, generated, verified):
        """Closing card: what this episode's topic is built from and checked by."""
        self.heading("Files", "what this part is written in, what is generated, and what proves it")
        cols = []
        for title, items, col in (("written by hand", inputs, C_RTL),
                                  ("generated", generated, C_PY),
                                  ("verified by", verified, C_BIT)):
            head = Text(title, font_size=21, color=col, weight="BOLD")
            cards = VGroup(*[filecard(p, r, col) for p, r in items])
            cards.arrange(DOWN, aligned_edge=LEFT, buff=0.18)
            g = VGroup(head, cards).arrange(DOWN, aligned_edge=LEFT, buff=0.28)
            cols.append(g)
        row = VGroup(*cols).arrange(RIGHT, buff=0.7, aligned_edge=UP)
        if row.width > 13.2:
            row.scale_to_fit_width(13.2)
        row.next_to(self.mobjects[1], DOWN, buff=0.55).set_x(0)
        for c in cols:
            self.play(FadeIn(c, shift=UP * 0.2), run_time=0.7)
        self.wait(2.4)

# =============================================================================
#  EPISODE 10 - Bitstream generation: FASM, bitgen, and the .bit container
# =============================================================================

def s1_fasm(sc):
    sc.heading("FASM: the seam between 'where things go' and 'which bits are set'",
               "borrowed from F4PGA / prjxray - a feature is a name and a value, nothing more")

    ex = code_block([
        "clb_x2y3.init      = 64'h8888888888888888",
        "clb_x2y3.ff_en     = 1'h1",
        "bram0.wmode_a      = 2'h1",
        "ctrl.clk_div       = 5'hF",
        "rr1204             = 3'h2",
    ], 24, C_BIT)
    exp = panel(ex, C_BIT)
    exp.shift(UP * 1.5)
    sc.play(FadeIn(exp), run_time=0.9)

    kinds = code_block([
        "<block>.<field>   a field of a CLB, BRAM or DSP block",
        "ctrl.<field>      the 8-bit ctrl tile - the user clock",
        "rr<node>          the routing mux of rr-graph node <node>",
        "",
        "only non-zero features need writing; everything else is 0",
    ], 20, INK)
    kinds[4].set_color(DIM)
    kinds.next_to(exp, DOWN, buff=0.8).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in kinds], lag_ratio=0.18), run_time=1.6)

    why = Text("Because it is text, the same file can come from VPR or from bob's own PnR, "
               "and you can read it, diff it and hand-edit it.",
               font_size=20, color=C_BIT)
    why.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.4)
    sc.play(FadeIn(why), run_time=0.9)
    sc.wait(2.0)


def s2_bitgen(sc):
    sc.heading("bitgen: FASM to bits, and back again exactly",
               "tools/bob/bitgen.py checks every feature against device.json before setting a bit")

    f = mono("clb_x2y3.ff_en = 1'h1", 22, C_BIT).shift(UP * 2.1)
    sc.play(FadeIn(f), run_time=0.5)

    steps = [
        ("look up the block", "device.json: clb_x2y3 is at chain_lo = 9412", C_PY),
        ("look up the field", "ff_en is offset 64, width 1", C_PY),
        ("check the width", "the declared 1'h must equal the device's", C_ERR),
        ("check the value", "fits the field; a mux value must be a real input", C_ERR),
        ("set the bit", "chain bit 9412 + 64 = 9476", C_RTL),
    ]
    g = VGroup()
    for a, b, col in steps:
        g.add(VGroup(mono(a, 20, col), Text(b, font_size=17, color=DIM))
              .arrange(RIGHT, buff=0.5, aligned_edge=DOWN))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 3.8)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.26)
    if g.width > 12.6:
        g.scale_to_fit_width(12.6)
    g.next_to(f, DOWN, buff=0.7).set_x(0)
    for r in g:
        sc.play(FadeIn(r, shift=RIGHT * 0.2), run_time=0.4)

    rt = code_block([
        "and the other direction: chain -> FASM decodes every configurable field",
        "and REFUSES any set bit that no feature owns - the padding, the tail.",
        "",
        "So chain -> FASM -> chain is exact.  bitgen.py --roundtrip proves it.",
    ], 20, INK)
    rt[3].set_color(C_BIT)
    rt.next_to(g, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in rt], lag_ratio=0.2), run_time=1.7)
    sc.wait(2.0)


def s3_bitfile(sc):
    sc.heading("The .bit container",
               "host-side only - the chip never sees a header. Chain file version 2.")

    rows = [
        ("BOBC", "magic", C_GRF),
        ("version = 2", "version 1 files still load", C_GRF),
        ("device name", "bob12x10 - a bitstream for the wrong device is refused", C_ERR),
        ("chain width W", "18560 - so is the wrong size", C_ERR),
        ("CRC-32C", "checked BEFORE any hardware is touched", C_ERR),
        ("the chain bytes", "byte j = chain bits [8j+7 : 8j]", C_BIT),
        ("section: BRAM", "index, first address, count, words - one per used BRAM", C_VPR),
        ("section: META", "JSON: design, top, source sha256s, pcf, PnR hash, clock", C_PY),
        ("file CRC-32C", "over every byte before it", C_ERR),
    ]
    g = VGroup()
    for a, b, col in rows:
        box = Rectangle(width=3.4, height=0.5, color=col, stroke_width=2)
        box.set_fill(col, opacity=0.16)
        t = mono(a, 17, INK)
        if t.width > 3.2:
            t.scale_to_fit_width(3.2)
        note = Text(b, font_size=16, color=DIM)
        g.add(VGroup(VGroup(box, t.move_to(box)), note)
              .arrange(RIGHT, buff=0.5, aligned_edge=LEFT))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 3.9)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.12)
    if g.height > 5.2:
        g.scale_to_fit_height(5.2)
    g.next_to(sc.mobjects[1], DOWN, buff=0.45).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=UP * 0.15) for r in g], lag_ratio=0.12),
            run_time=2.2)

    note = Text("Three independent refusals before a single TCK pulse: wrong device, "
                "wrong width, corrupt file.", font_size=19, color=C_BIT)
    note.scale_to_fit_width(12.8).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(note), run_time=0.8)
    sc.wait(2.0)


def s4_load(sc):
    sc.heading("./bob load", "what actually happens between the file and a lit LED")

    steps = [
        "check the file: magic, device, width, CRC",
        "JPROGRAM  - clear the memory, drop DONE",
        "send the frames on CFG_IN, BRAM contents in the same CRC-covered stream",
        "read STAT: START accepted, no error",
        "FDRO readback of the whole memory - must equal the .bit",
        "JSTART + 12 TCK in Run-Test/Idle",
        "read DONE",
    ]
    g = VGroup()
    for i, s in enumerate(steps):
        n = mono(f"{i + 1}.", 20, C_BIT)
        t = Text(s, font_size=19, color=INK)
        g.add(VGroup(n, t).arrange(RIGHT, buff=0.35, aligned_edge=DOWN))
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.3)
    if g.width > 12.6:
        g.scale_to_fit_width(12.6)
    g.next_to(sc.mobjects[1], DOWN, buff=0.7).set_x(0)
    for r in g:
        sc.play(FadeIn(r, shift=RIGHT * 0.2), run_time=0.4)

    alt = code_block([
        "./bob load x.bit                the frame path (default)",
        "./bob load x.bit --mode chain   the same memory, one giant scan + USER4",
        "./bob load x.bit --partial      only the frames that differ, design keeps running",
    ], 19, C_PY)
    alt.next_to(g, DOWN, buff=0.7).set_x(0)
    sc.play(FadeIn(alt), run_time=0.9)

    ver = Text("Every load is verified by reading the memory back. A load that cannot be "
               "read back is a failed load, not a warning.",
               font_size=19, color=C_BIT)
    ver.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(ver), run_time=0.8)
    sc.wait(2.0)


def s5_files(sc):
    sc.files_used(
        inputs=[("tools/bob/bitgen.py", "FASM <-> chain, .bit reader and writer"),
                ("tools/bob/fasm_from_vpr.py", "PnR result -> FASM features"),
                ("tools/bob/cli.py", "./bob build | load | info | fasm")],
        generated=[("build/bit/<name>.bit", "the loadable bitstream"),
                   ("the FASM text", "./bob fasm x.bit prints it back out"),
                   ("docs/reports/M11/designs.md", "per-design cells, CLBs, CRC")],
        verified=[("tests/test_bitgen.py", "round trip, bad features rejected"),
                  ("sim/tb_cosim.v", "the real .bit loaded into the full FPGA RTL"),
                  ("docs/hwtest/results.log", "bob-* checks: readback == FASM on the board")])


EP10 = [s1_fasm, s2_bitgen, s3_bitfile, s4_load, s5_files]


class Ep10Bitgen(BobScene):
    def construct(self):
        self.titlecard("EPISODE 10", "Bitstream generation",
                       "FASM, bitgen, and the container that refuses bad files")
        for i, part in enumerate(EP10):
            part(self)
            if i < len(EP10) - 1:
                clear_all(self)


class E10S1Fasm(BobScene):
    def construct(self): s1_fasm(self)


class E10S2Bitgen(BobScene):
    def construct(self): s2_bitgen(self)


class E10S3Bitfile(BobScene):
    def construct(self): s3_bitfile(self)


class E10S4Load(BobScene):
    def construct(self): s4_load(self)


class E10S5Files(BobScene):
    def construct(self): s5_files(self)